In [0]:
dbutils.widgets.dropdown(
    "taxi_type",
    "yellow",
    ["yellow", "green"],
    "Taxi type"
)

dbutils.widgets.text("year", "2025", "Year")
dbutils.widgets.text("month", "1", "Month")

In [0]:
from datetime import datetime, timezone

taxi_type = dbutils.widgets.get("taxi_type").strip().lower()
year = int(dbutils.widgets.get("year"))
month = int(dbutils.widgets.get("month"))

now_utc = datetime.now(timezone.utc)

if taxi_type not in {"yellow", "green"}:
    raise ValueError("taxi_type must be yellow or green.")

if year < 2009:
    raise ValueError("year cannot be earlier than 2009.")

if not 1 <= month <= 12:
    raise ValueError("month must be between 1 and 12.")

if (year, month) > (now_utc.year, now_utc.month):
    raise ValueError("The requested period is in the future.")

print(f"Processing {taxi_type} taxi data for {year}-{month:02d}")

In [0]:
period = f"{year}-{month:02d}"

file_name = f"{taxi_type}_tripdata_{period}.parquet"

bronze_base_path = (
    "/Volumes/workspace/urban_mobility_bronze/landing"
)

source_path = (
    f"{bronze_base_path}/{taxi_type}/"
    f"year={year}/month={month:02d}/"
    f"{file_name}"
)

print(f"File name: {file_name}")
print(f"Source path: {source_path}")




In [0]:
from pathlib import Path

if not Path(source_path).is_file():
    raise FileNotFoundError(
        f"Bronze source file does not exist: {source_path}"
    )

In [0]:
from pyspark.sql.functions import lit, current_timestamp 

raw_df = spark.read.parquet(source_path)

print(f"Raw rows: {raw_df.count():,}")
print(f"Raw columns: {len(raw_df.columns)}")
display(raw_df.limit(10))


# rename columns and add new columns
canonical_df = raw_df.withColumn("taxi_type", lit(taxi_type)).withColumn("year",lit(year)).withColumn("month",lit(month)).withColumn("source_path",lit(source_path)).withColumn("processed_at",lit(current_timestamp())).withColumnsRenamed({
    "VendorID": "vendor_id",
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "PULocationID" : "pickup_location_id",
    "DOLocationID" : "dropoff_location_id"
})

# display(canonical_df)

# canonical_df.printSchema()
# display(canonical_df.limit(10))

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

period_start = datetime(year, month, 1)

if month == 12:
    period_end = datetime(year + 1, 1, 1)
else:
    period_end = datetime(year, month + 1, 1)

print(f"Period start: {period_start}")
print(f"Period end: {period_end}")


quality_df = canonical_df.withColumn(
    "dq_reasons",
    F.array_compact(
        F.array(
            F.when(
                F.col("pickup_datetime").isNull(),
                F.lit("MISSING_PICKUP_DATETIME")
            ),

            F.when(
                F.col("dropoff_datetime").isNull(),
                F.lit("MISSING_DROPOFF_DATETIME")
            ),

            F.when(
                F.col("dropoff_datetime")
                <= F.col("pickup_datetime"),
                F.lit("INVALID_TRIP_DURATION")
            ),

            F.when(
                (F.col("pickup_datetime") < F.lit(period_start))
                | (F.col("pickup_datetime") >= F.lit(period_end)),
                F.lit("PICKUP_OUTSIDE_SOURCE_MONTH")
            ),

            F.when(
                F.col("trip_distance").isNull()
                | (F.col("trip_distance") <= 0)
                | (F.col("trip_distance") > 200),
                F.lit("INVALID_TRIP_DISTANCE")
            ),

            F.when(
                F.col("pickup_location_id").isNull()
                | ~F.col("pickup_location_id").between(1, 265),
                F.lit("INVALID_PICKUP_LOCATION")
            ),

            F.when(
                F.col("dropoff_location_id").isNull()
                | ~F.col("dropoff_location_id").between(1, 265),
                F.lit("INVALID_DROPOFF_LOCATION")
            ),

            F.when(
                F.col("passenger_count").isNotNull()
                & ~F.col("passenger_count").between(1, 8),
                F.lit("INVALID_PASSENGER_COUNT")
            ),

            F.when(
                F.col("fare_amount").isNull()
                | (F.col("fare_amount") < 0),
                F.lit("INVALID_FARE_AMOUNT")
            ),

            F.when(
                F.col("total_amount").isNull()
                | (F.col("total_amount") < 0),
                F.lit("INVALID_TOTAL_AMOUNT")
            )
        )
    )
)

In [0]:
valid_df = (
    quality_df
    .filter(F.size("dq_reasons") == 0)
    .drop("dq_reasons")
)

quarantine_df = (
    quality_df
    .filter(F.size("dq_reasons") > 0)
)
quality_summary = (
    quality_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(F.size("dq_reasons") == 0, 1)
            .otherwise(0)
        ).alias("valid_rows"),

        F.sum(
            F.when(F.size("dq_reasons") > 0, 1)
            .otherwise(0)
        ).alias("quarantine_rows")
    )
    .first()
)

total_rows = quality_summary["total_rows"]
valid_rows = quality_summary["valid_rows"]
quarantine_rows = quality_summary["quarantine_rows"]

print(f"Total rows: {total_rows:,}")
print(f"Valid rows: {valid_rows:,}")
print(f"Quarantine rows: {quarantine_rows:,}")
print(
    f"Valid percentage: "
    f"{valid_rows / total_rows * 100:.2f}%"
)


reason_summary_df = (
    quarantine_df
    .select(F.explode("dq_reasons").alias("dq_reason"))
    .groupBy("dq_reason")
    .count()
    .orderBy(F.desc("count"))
)

display(reason_summary_df)

In [0]:
trip_identity_columns = [
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "taxi_type",
]

identified_df = quality_df.withColumn(
    "trip_id",
    F.sha2(                     # provide a unique hash for each trip
        F.to_json(              # convert the trip identity columns to a JSON string
            F.struct(           # create a struct with the trip identity columns
                *[
                    F.col(column_name)
                    for column_name in trip_identity_columns
                ]
            )
        ),
        256
    )
)

# drop duplicates
deduplicated_df = identified_df.dropDuplicates(["trip_id"])

#separate the valid and quarantine dataframes
silver_valid_df = (
    deduplicated_df
    .filter(F.size("dq_reasons") == 0)
    .drop("dq_reasons")
)

silver_quarantine_df = (
    deduplicated_df
    .filter(F.size("dq_reasons") > 0)
)

#calculate deduplication stats
dedup_summary = (
    deduplicated_df
    .agg(
        F.count("*").alias("rows_after_deduplication"),

        F.sum(
            F.when(F.size("dq_reasons") == 0, 1)
            .otherwise(0)
        ).alias("valid_rows_after_deduplication"),

        F.sum(
            F.when(F.size("dq_reasons") > 0, 1)
            .otherwise(0)
        ).alias("quarantine_rows_after_deduplication")
    )
    .first()
)

rows_after_deduplication = dedup_summary[
    "rows_after_deduplication"
]

duplicate_rows = total_rows - rows_after_deduplication

print(f"Input rows: {total_rows:,}")
print(f"Rows after deduplication: {rows_after_deduplication:,}")
print(f"Exact duplicates removed: {duplicate_rows:,}")
print(
    "Valid after deduplication: "
    f"{dedup_summary['valid_rows_after_deduplication']:,}"
)
print(
    "Quarantine after deduplication: "
    f"{dedup_summary['quarantine_rows_after_deduplication']:,}"
)
#check uniqueness of trip id
duplicate_trip_ids = (
    silver_valid_df
    .groupBy("trip_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_trip_ids != 0:
    raise ValueError(
        "Duplicate trip_id values remain after deduplication."
    )

print("trip_id uniqueness validation passed.")

In [0]:
# define table and partition
valid_table = (
    "workspace.urban_mobility_silver.taxi_trips"
)

quarantine_table = (
    "workspace.urban_mobility_silver.taxi_trips_quarantine"
)

partition_columns = [
    "taxi_type",
    "year",
    "month",
]

replace_predicate = (
    f"taxi_type = '{taxi_type}' "
    f"AND year = {year} "
    f"AND month = {month}"
)

print(f"Replace predicate: {replace_predicate}")

# write a reusable function
def write_delta_partition(
    dataframe,
    table_name,
    replace_condition
):
    writer = dataframe.write.format("delta")

    if spark.catalog.tableExists(table_name):
        (
            writer
            .mode("overwrite")
            .option("replaceWhere", replace_condition)
            .saveAsTable(table_name)
        )

        print(
            f"Replaced partition in existing table: "
            f"{table_name}"
        )

    else:
        (
            writer
            .mode("overwrite")
            .partitionBy(*partition_columns)
            .saveAsTable(table_name)
        )

        print(f"Created Delta table: {table_name}")

write_delta_partition(
    silver_valid_df,
    valid_table,
    replace_predicate
)

write_delta_partition(
    silver_quarantine_df,
    quarantine_table,
    replace_predicate
)        



In [0]:
%sql
SELECT
    'valid' AS dataset,
    COUNT(*) AS row_count
FROM workspace.urban_mobility_silver.taxi_trips
WHERE taxi_type = 'yellow'
  AND year = 2025
  AND month = 1

UNION ALL

SELECT
    'quarantine' AS dataset,
    COUNT(*) AS row_count
FROM workspace.urban_mobility_silver.taxi_trips_quarantine
WHERE taxi_type = 'yellow'
  AND year = 2025
  AND month = 1;

In [0]:
%sql
DESCRIBE HISTORY workspace.urban_mobility_silver.taxi_trips;

In [0]:
from uuid import uuid4
from datetime import datetime, timezone

silver_run_id = uuid4().hex
silver_started_at = datetime.now(timezone.utc)

print(f"Silver run ID: {silver_run_id}")
print(f"Started at: {silver_started_at.isoformat()}")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.urban_mobility_ops.silver_transformation_runs (
    run_id STRING NOT NULL,
    taxi_type STRING NOT NULL,
    data_year INT NOT NULL,
    data_month INT NOT NULL,
    source_path STRING NOT NULL,
    started_at TIMESTAMP NOT NULL,
    completed_at TIMESTAMP NOT NULL,
    status STRING NOT NULL,
    input_rows BIGINT NOT NULL,
    rows_after_deduplication BIGINT NOT NULL,
    duplicate_rows BIGINT NOT NULL,
    valid_rows BIGINT NOT NULL,
    quarantine_rows BIGINT NOT NULL,
    valid_percentage DOUBLE NOT NULL,
    error_message STRING
)
USING DELTA
COMMENT 'Audit history for Silver taxi trip transformations';

In [0]:
silver_completed_at = datetime.now(timezone.utc)
silver_status = "SUCCESS"

valid_rows_after_deduplication = int(
    dedup_summary["valid_rows_after_deduplication"]
)

quarantine_rows_after_deduplication = int(
    dedup_summary["quarantine_rows_after_deduplication"]
)

valid_percentage_after_deduplication = (
    valid_rows_after_deduplication
    / rows_after_deduplication
    * 100
)

silver_audit_df = (
    spark.range(1)
    .select(
        F.lit(silver_run_id)
            .alias("run_id"),

        F.lit(taxi_type)
            .alias("taxi_type"),

        F.lit(year)
            .cast("int")
            .alias("data_year"),

        F.lit(month)
            .cast("int")
            .alias("data_month"),

        F.lit(source_path)
            .alias("source_path"),

        F.lit(silver_started_at)
            .cast("timestamp")
            .alias("started_at"),

        F.lit(silver_completed_at)
            .cast("timestamp")
            .alias("completed_at"),

        F.lit(silver_status)
            .alias("status"),

        F.lit(total_rows)
            .cast("long")
            .alias("input_rows"),

        F.lit(rows_after_deduplication)
            .cast("long")
            .alias("rows_after_deduplication"),

        F.lit(duplicate_rows)
            .cast("long")
            .alias("duplicate_rows"),

        F.lit(valid_rows_after_deduplication)
            .cast("long")
            .alias("valid_rows"),

        F.lit(quarantine_rows_after_deduplication)
            .cast("long")
            .alias("quarantine_rows"),

        F.lit(valid_percentage_after_deduplication)
            .cast("double")
            .alias("valid_percentage"),

        F.lit(None)
            .cast("string")
            .alias("error_message")
    )
)

silver_audit_df.write.mode("append").saveAsTable(
    "workspace.urban_mobility_ops.silver_transformation_runs"
)

print(f"Silver audit saved: {silver_run_id}")

In [0]:
%sql
SELECT *
FROM workspace.urban_mobility_ops.silver_transformation_runs
ORDER BY started_at DESC;

In [0]:
%sql
SELECT
    run_id,
    input_rows,
    valid_rows,
    quarantine_rows,
    duplicate_rows,
    input_rows = (
        valid_rows + quarantine_rows + duplicate_rows
    ) AS counts_reconciled,
    completed_at >= started_at AS timestamps_valid
FROM workspace.urban_mobility_ops.silver_transformation_runs
ORDER BY started_at DESC;